# Qwen Image 2.1 Turbo on Kaggle (ComfyUI)

部署 ComfyUI（最新 master，原生支持 Qwen Image 2.1）+ Qwen Image 2.1 Turbo int8 + Qwen3-VL 8B int8 编码器 + Qwen Image 2.1 VAE，跑一张 1024×1024 文生图测试。

模型来自 `Comfy-Org/Qwen-Image-2.1`（公开仓库，HF API 已核对：private false、gated false，无需 token）：Turbo int8 DiT 6.76GB、Qwen3-VL 8B int8 编码器 8.71GB、VAE bf16 0.64GB，合计约 16.1GB，装得进 /kaggle/working 20GB。两个提示词增强器（pe_t2i / pe_i2i，各约 8.8GB）不下载——官方模板默认 refine_prompt 就是关的，提示词原样送入模型。

工作流按官方 Turbo T2I 模板（Comfy-Org/workflow_templates，image_qwen_image_2_1_turbo_t2i.json）展开成原始节点提交，不直接提交子图：子图里含整条提示词增强器分支（TextGenerate + PE 编码器），即使 refine 关着也会加载 PE 模型；且 /prompt API 只接受原始节点，子图是前端概念。Turbo 采样 8 步、cfg 1、euler + 固定 ManualSigmas 排程。

注意：ComfyUI 服务由出图那格用 subprocess.Popen 在本进程内启动（Kaggle 版本运行里跨格的 nohup 后台进程活不下来），跑完即停。

LoRA 预留：下载格和出图格里各有一段 LoRA 配置（LORA_NAME / LORA_REPO / LORA_STRENGTH，默认空）。需要加 LoRA 时填上文件名和仓库，下载格会把它下到 ComfyUI/models/loras/，出图格会在 UNETLoader 后接一个 LoraLoaderModelOnly 节点（只改模型，不动 CLIP）；留空时节点不接入，模型线完全不变。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
!ls /kaggle/working/ComfyUI | head -20

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
# NOTE: filenames must include the repo subdirectory; hf_hub_download places the file at local_dir/filename.
#
# LoRA (optional, reserved for later use): set LORA_NAME to a .safetensors file
# in ComfyUI/models/loras/ and LORA_REPO to the Hugging Face repo hosting it
# (file assumed at the repo root). Leave LORA_NAME = "" (default) to run
# without a LoRA. When set, the file is downloaded here and the run cell wires
# a LoraLoaderModelOnly node into the model line.
LORA_NAME = ""
LORA_REPO = ""
LORA_STRENGTH = 1.0
# Repo paths ARE diffusion_models/ / text_encoders/ / vae/ (no split_files layer, unlike Wan's repo).
# Sizes from the official template's model note (T2I template MarkdownNote "Model Links"):
# DiT 6.76 GB, encoder 8.71 GB, VAE 644.2 MB. Total ~16.1 GB, fits /kaggle/working 20 GB.
jobs = [
    "diffusion_models/qwen_image_2.1_turbo_int8_convrot.safetensors",
    "text_encoders/qwen3vl_8b_int8_convrot.safetensors",
    "vae/qwen_image_2.1_vae_bf16.safetensors",
]
for fname in jobs:
    p = hf_hub_download(repo_id="Comfy-Org/Qwen-Image-2.1", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": "qwen_image_2.1_turbo_int8_convrot.safetensors",
    "text_encoders": "qwen3vl_8b_int8_convrot.safetensors",
    "vae": "qwen_image_2.1_vae_bf16.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
# Negative prompt: template ships "" and docs say at cfg 1 ComfyUI skips the negative
# conditioning pass, so a negative prompt has no effect (docs.comfy.org Qwen-Image-2.1, "Sampler settings").
NEGATIVE = ""

# Turbo 8-step sigmas, verbatim from the official T2I template:
# - node 484 ManualSigmas, title "Manual Sigmas (Qwen-Image-2.1-Turbo 8-step schedule)",
#   widgets_values_named.sigmas (template line ~2816)
# - template Usage note: "the schedule is fixed in the ManualSigmas node (1.0, 0.978453,
#   0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0) and is what makes 8 steps enough"
# - docs: "The two Turbo templates sample in 8 steps with cfg 1, the euler sampler, and the
#   fixed ManualSigmas schedule that ships with the distilled checkpoint"
SIGMAS = "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0"

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)
need = ["UNETLoader", "CLIPLoader", "VAELoader", "TextEncodeQwenImage21", "EmptyLatentImage",
        "QwenImage21Cache", "CFGGuider", "KSamplerSelect", "RandomNoise", "ManualSigmas",
        "SamplerCustomAdvanced", "VAEDecode", "SaveImage"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", {}))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the specs we are least sure about, so the log shows the truth (and read them
# before the next submission -- the Wan V5 lesson: printing a spec without reading it
# does not count as checking it).
for ct in ["TextEncodeQwenImage21", "QwenImage21Cache", "SamplerCustomAdvanced", "ManualSigmas", "RandomNoise"]:
    print(ct, "spec:", json.dumps(info[ct]["input"], ensure_ascii=False)[:800])

# Node wiring, reconstructed from the official Turbo T2I template subgraph
# (definitions.subgraphs[0], id c291ceec-b98f-4751-9d0b-7bc288f27b30). Template node ids in brackets.
nodes = {}
# [451] UNETLoader: widgets_values_named unet_name / weight_dtype
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "qwen_image_2.1_turbo_int8_convrot.safetensors", "weight_dtype": "default"})}
# [453] CLIPLoader: clip_name / type "qwen_image" / device "default"
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image", "device": "default"})}
# [454] VAELoader: vae_name
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"})}
# [452] TextEncodeQwenImage21: clip (link 647), prompt, negative_prompt, resolution 1024
# (widgets_values_named). images.image_1 / vae inputs are unconnected in T2I, so omitted.
# resolution is the edit canvas budget (docs "Resolution"); with no reference images in
# T2I it has nothing to resize -- we pass the template's value as-is.
# images is a REQUIRED autogrow input (COMFY_AUTOGROW_V3) in current master: it is a
# list of image links, and pure T2I means zero reference images, i.e. an empty list.
# V1 failed here because the input was omitted entirely and the pre-check flagged it
# as missing. The official T2I template runs with no images connected, so [] is valid.
nodes["4"] = {"class_type": "TextEncodeQwenImage21", "inputs": node_inputs("TextEncodeQwenImage21", {"clip": ["2", 0], "prompt": PROMPT, "negative_prompt": NEGATIVE, "resolution": 1024, "images": []})}
# [456] EmptyLatentImage: width / height / batch_size
nodes["5"] = {"class_type": "EmptyLatentImage", "inputs": node_inputs("EmptyLatentImage", {"width": 1024, "height": 1024, "batch_size": 1})}
# [490] QwenImage21Cache: device "auto", dtype "default" (docs "KV cache" table:
# device auto/gpu/cpu/off, dtype default/int8/int4). Wired UNETLoader -> Cache -> CFGGuider.
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) between UNETLoader and the cache node. When empty, the
# cache takes the model straight from UNETLoader, unchanged.
MODEL_OUT = ["1", 0]
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    MODEL_OUT = ["20", 0]
nodes["6"] = {"class_type": "QwenImage21Cache", "inputs": node_inputs("QwenImage21Cache", {"model": MODEL_OUT, "device": "auto", "dtype": "default"})}
# [489] CFGGuider: model / positive / negative / cfg 1 (template widgets_values_named cfg 1;
# docs: "keep 1 for the Qwen Image 2.1 official path")
nodes["7"] = {"class_type": "CFGGuider", "inputs": node_inputs("CFGGuider", {"model": ["6", 0], "positive": ["4", 0], "negative": ["4", 1], "cfg": 1})}
# [487] KSamplerSelect: sampler_name "euler" (template; docs: euler)
nodes["8"] = {"class_type": "KSamplerSelect", "inputs": node_inputs("KSamplerSelect", {"sampler_name": "euler"})}
# [488] RandomNoise: template ships noise_seed 531123706903762 / control_after_generate
# "randomize", and the subgraph's exposed "seed" input does NOT feed this node -- link 697
# goes only to TextGenerate.sampling_mode.seed (the PE branch). For a reproducible first
# test we use seed 42 / "fixed"; the combo pre-check below validates "fixed" against the
# live node definition before anything is submitted.
nodes["9"] = {"class_type": "RandomNoise", "inputs": node_inputs("RandomNoise", {"noise_seed": 42, "control_after_generate": "fixed"})}
# [484] ManualSigmas: sigmas (see SIGMAS above)
nodes["10"] = {"class_type": "ManualSigmas", "inputs": node_inputs("ManualSigmas", {"sigmas": SIGMAS})}
# [486] SamplerCustomAdvanced: noise (715) / guider (716) / sampler (717) / sigmas (718) / latent_image (726)
nodes["11"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["9", 0], "guider": ["7", 0], "sampler": ["8", 0], "sigmas": ["10", 0], "latent_image": ["5", 0]}}
# [457] VAEDecode: samples (720) / vae (649)
nodes["12"] = {"class_type": "VAEDecode", "inputs": {"samples": ["11", 0], "vae": ["3", 0]}}
# SaveImage (core node, as in the MOSS notebook; the template uses SaveImageAdvanced with
# prefix "Qwen_image_2.1" -- same output, simpler API surface for the test).
nodes["13"] = {"class_type": "SaveImage", "inputs": {"images": ["12", 0], "filename_prefix": "qwen_test"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: dynamic combos (COMFY_DYNAMICCOMBO_V3) expose cfg[0] as a string,
# not a list, so they are skipped by this check and still need manual verification.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
print("submit:", r.status_code, r.text[:400])
r.raise_for_status()
pid = r.json()["prompt_id"]
print("prompt_id:", pid)

out_path = "/kaggle/working/qwen_test.png"
t0 = time.time()
done = False
while time.time() - t0 < 1800:
    h = requests.get(BASE + "/history/" + pid, timeout=60).json()
    if pid in h:
        st = h[pid].get("status", {})
        sstr = st.get("status_str")
        if sstr == "success":
            outs = h[pid]["outputs"]
            imgs = outs.get("13", {}).get("images", [])
            print("images:", imgs)
            if imgs:
                im = imgs[0]
                q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                data = requests.get(BASE + "/view", params=q, timeout=300).content
                open(out_path, "wb").write(data)
                print("saved", out_path, len(data), "bytes")
                done = True
            break
        if sstr == "error":
            print("RUN ERROR:", json.dumps(h[pid], indent=2)[:3000])
            break
    time.sleep(15)
print("done:", done, "elapsed:", round(time.time() - t0), "s")
proc.terminate()
print("server stopped")

In [ ]:
from IPython.display import Image, display
import os
p = "/kaggle/working/qwen_test.png"
print("exists:", os.path.exists(p), "bytes:", os.path.getsize(p) if os.path.exists(p) else 0)
display(Image(filename=p))